# Alfamedomega training on Colab

Runs `alfamedomega_train.py` (method: `METHOD.md`). Use a GPU runtime: a T4 is enough for Lite, and Studio wants an A100.
Checkpoints, example audio and exports go to Google Drive so a disconnected runtime loses nothing.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > GPU"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/alfamedomega'   # change if you like
import os; os.makedirs(OUT, exist_ok=True)

Put `alfamedomega_train.py` in the Drive folder above, or upload it when this cell asks.

In [ ]:
import os, shutil
src = os.path.join(OUT, 'alfamedomega_train.py')
if os.path.exists(src):
    shutil.copy(src, '/content/alfamedomega_train.py')
else:
    from google.colab import files
    files.upload()   # pick alfamedomega_train.py
!pip -q install onnx onnxruntime
!python /content/alfamedomega_train.py report

## 1. Smoke test (about a minute)
Checks every stage of the loop end to end before spending GPU hours.

In [ ]:
!python /content/alfamedomega_train.py train --tier lite --smoke --out /content/smoke
!python /content/alfamedomega_train.py export --ckpt /content/smoke/lite_pretrain.pt --out /content/smoke/Models

## 2. Lite pretrain
The first real run. Watch `mrstft` fall in the log, and listen to the `*_pred.wav` / `*_target.wav` pairs it writes. Re-running with `--init` continues from the checkpoint.

In [ ]:
!python /content/alfamedomega_train.py train --tier lite --steps 20000 --out $OUT/lite --audio_every 2000 --ckpt_every 1000

In [ ]:
import glob, IPython.display as ipd
for f in sorted(glob.glob(f'{OUT}/lite/*.wav'))[-4:]:
    print(f); ipd.display(ipd.Audio(f))

## 3. Studio: pretrain, then polish (A100)
Studio is the reference the smaller tiers are distilled from.

In [ ]:
!python /content/alfamedomega_train.py train --tier studio --steps 200000 --out $OUT/studio
!python /content/alfamedomega_train.py train --tier studio --stage polish --steps 50000 --init $OUT/studio/studio_pretrain.pt --out $OUT/studio

## 4. Real-audio fine-tune (NSynth)
NSynth is CC BY 4.0. The validation split is the smallest download; use the train split for real runs.

In [ ]:
!cd /content && wget -q http://download.magenta.tensorflow.org/datasets/nsynth/nsynth-valid.jsonwav.tar.gz && tar xzf nsynth-valid.jsonwav.tar.gz
!python /content/alfamedomega_train.py train --tier studio --stage finetune --data /content/nsynth-valid --steps 50000 --init $OUT/studio/studio_polish.pt --out $OUT/studio

## 5. Distil Pro, Standard and Lite from Studio

In [ ]:
for tier in ['pro', 'standard', 'lite']:
    !python /content/alfamedomega_train.py train --tier {tier} --stage distill --teacher $OUT/studio/studio_finetune.pt --data /content/nsynth-valid --steps 100000 --out $OUT/{tier}

## 6. Export for the plugin
Writes the ONNX graphs, `grain_bank.npy`, `manifest.json` and `dsp_golden.npz`, and fails if streaming or ONNX Runtime parity breaks.

In [ ]:
!python /content/alfamedomega_train.py export --ckpt $OUT/lite/lite_distill.pt --out $OUT/Models --int8
for tier in ['standard', 'pro']:
    !python /content/alfamedomega_train.py export --ckpt $OUT/{tier}/{tier}_distill.pt --out $OUT/Models
!python /content/alfamedomega_train.py export --ckpt $OUT/studio/studio_finetune.pt --out $OUT/Models